# Analisis Exploratorio de la informacion incial

In [195]:
#Importamos las librerias necesarias para el desarrollo
import pandas as pd

## Lectura inicial para la lectura de la informacion:

* clientes
* detalle pedidos
* eventos
* pedidos
* productos

In [196]:
#Lectura de la información
df_clientes = pd.read_csv("../data/raw/clientes.csv")
df_detalle_pedidos = pd.read_csv("../data/raw/detalle_pedidos.csv")
df_eventos = pd.read_csv("../data/raw/eventos.csv")
df_pedidos = pd.read_csv("../data/raw/pedidos.csv")
df_productos = pd.read_csv("../data/raw/productos.csv")

Diccionario para agilizar la obtencion del nombre de la tabla, el dataFrame correspondiente y la llave primaria

In [197]:
# Definimos las tablas y sus Primary Keys según el diccionario de datos
config_tablas = {
    "clientes": {"df": df_clientes, "pk": "cliente_id"},
    "pedidos": {"df": df_pedidos, "pk": "pedido_id"},
    "detalle_pedidos": {"df": df_detalle_pedidos, "pk": "item_id"},
    "productos": {"df": df_productos, "pk": "producto_id"},
    "eventos": {"df": df_eventos, "pk": "evento_id"},
}

# Funcion para el analisis inicial en este hacemos un conteo por Primary Key, para encontrar duplicados y campo por campo hacemos un conteo de nulos y el porcentaje correspondiente

In [198]:
def auditoria_avanzada(config):
    for nombre, info in config.items():
        df = info["df"]
        pk = info["pk"]

        total = len(df)
        #Validacion por Primary Key para el conteo de duplicados
        dup_id = df[pk].duplicated().sum()

        print(f"=== TABLA: {nombre.upper()} ===")
        print(f"Registros Totales: {total}")
        print(f"Duplicados de ID (Llave Primaria '{pk}'): {dup_id}")

        # Reporte de Nulos y Únicos
        reporte = pd.DataFrame({
            'Tipo': df.dtypes,
            'Nulos': df.isnull().sum(),
            '% Nulos': (df.isnull().sum() / total * 100).round(2),
            'Únicos': [df[col].nunique() for col in df.columns]
        })
        print(reporte)
        print("-" * 60 + "\n")

In [199]:
auditoria_avanzada(config_tablas)

=== TABLA: CLIENTES ===
Registros Totales: 300
Duplicados de ID (Llave Primaria 'cliente_id'): 0
                        Tipo  Nulos  % Nulos  Únicos
cliente_id            object      0     0.00     300
nombre                object      0     0.00     178
apellido              object      0     0.00     150
email                 object     12     4.00     288
telefono              object     25     8.33     275
ciudad                object      7     2.33     243
pais                  object      0     0.00       6
segmento              object      0     0.00       4
fecha_registro        object      0     0.00     248
fecha_consentimiento  object      0     0.00     243
activo                  bool      0     0.00       2
data_owner            object      0     0.00       2
clasificacion_dato    object      0     0.00       1
------------------------------------------------------------

=== TABLA: PEDIDOS ===
Registros Totales: 1200
Duplicados de ID (Llave Primaria 'pedido_id'): 21
  

# Funcion para validar la integridad del cruce de las tablas, para identificar si existe informacion que no exista en las tablas maestras

In [200]:
def validar_integridad_referencial(config):
    print("INTEGRIDAD REFERENCIAL")
    print("="*60)

    # 1. Mapeo de llaves maestras (Entramos a ['df'] para sacar la columna)
    maestros = {
    "clientes": set(config['clientes']['df'][config['clientes']['pk']]),
    "productos": set(config['productos']['df'][config['productos']['pk']]),
    "pedidos": set(config['pedidos']['df'][config['pedidos']['pk']])
    }
    resultados = []

    # Extraemos los DataFrames para que el código sea más limpio abajo
    df_pedidos = config['pedidos']['df']
    df_detalle = config['detalle_pedidos']['df']
    df_eventos = config['eventos']['df']

    # --- VALIDACIÓN 1: Pedidos -> Clientes ---
    ped_huerfanos = df_pedidos[~df_pedidos['cliente_id'].isin(maestros['clientes'])]
    resultados.append(["Pedidos", "cliente_id", "Clientes", len(ped_huerfanos)])

    # --- VALIDACIÓN 2: Detalle_Pedidos -> Pedidos ---
    det_huerfanos_ped = df_detalle[~df_detalle['pedido_id'].isin(maestros['pedidos'])]
    resultados.append(["Detalle_Pedidos", "pedido_id", "Pedidos", len(det_huerfanos_ped)])

    # --- VALIDACIÓN 3: Detalle_Pedidos -> Productos ---
    det_huerfanos_prod = df_detalle[~df_detalle['producto_id'].isin(maestros['productos'])]
    resultados.append(["Detalle_Pedidos", "producto_id", "Productos", len(det_huerfanos_prod)])

    # --- VALIDACIÓN 4: Eventos -> Clientes (Solo los no nulos) ---
    evt_con_id = df_eventos[df_eventos['cliente_id'].notnull()]
    evt_huerfanos = evt_con_id[~evt_con_id['cliente_id'].isin(maestros['clientes'])]
    resultados.append(["Eventos", "cliente_id", "Clientes", len(evt_huerfanos)])

    # Crear tabla de resultados
    df_res = pd.DataFrame(resultados, columns=["Tabla Origen", "Columna FK", "Tabla Destino (PK)", "Huérfanos"])

    print(df_res)

    # Alerta de criticidad
    total_error = df_res['Huérfanos'].sum()
    if total_error > 0:
        print(f"\nCRÍTICO: Se detectaron {total_error} registros sin padre.")
    else:
        print("\nINTEGRIDAD PERFECTA: Todas las llaves coinciden.")
    print("="*60 + "\n")

In [201]:
validar_integridad_referencial(config_tablas)

INTEGRIDAD REFERENCIAL
      Tabla Origen   Columna FK Tabla Destino (PK)  Huérfanos
0          Pedidos   cliente_id           Clientes          0
1  Detalle_Pedidos    pedido_id            Pedidos          0
2  Detalle_Pedidos  producto_id          Productos          0
3          Eventos   cliente_id           Clientes          0

INTEGRIDAD PERFECTA: Todas las llaves coinciden.



# Validamos los valores numericos de las tablas para la verificación de valores negativos, posibles variaciones o informacion que podria generar ruido, es decir cualquier valor que se considere fuera de la naturaleza de los datos

In [202]:
def analizar_estadisticas_numericas(config):
    print("ANÁLISIS ESTADÍSTICO Y DETECCIÓN DE ANOMALÍAS")
    print("="*60)

    for nombre_tabla, info in config.items():
        df = info['df']
        # Seleccionamos solo columnas numéricas (int y float)
        cols_numericas = df.select_dtypes(include=['number']).columns

        if len(cols_numericas) > 0:
            print(f"\nTablas: {nombre_tabla.upper()}")
            # 1. Obtener estadísticas básicas
            stats = df[cols_numericas].describe()
            print(stats)

            # 2. Búsqueda de Anomalías Críticas
            for col in cols_numericas:
                # Caso A: Valores negativos (No debería haber precios ni cantidades negativas)
                negativos = df[df[col] < 0]
                if not negativos.empty:
                    print(f"ALERTA: '{col}' tiene {len(negativos)} valores negativos.")

                # Caso B: Descuentos mayores al 100% (si la columna es de descuento)
                if 'descuento' in col.lower():
                    fuera_rango = df[df[col] > 1.0] # Si es porcentaje 0-1
                    if fuera_rango.empty: # Intenta con 100 si es de 0-100
                        fuera_rango = df[df[col] > 100]

                    if not fuera_rango.empty:
                        print(f"ALERTA: '{col}' tiene descuentos superiores al máximo permitido.")

                # Caso C: Stock en cero (Informativo)
                if 'stock' in col.lower():
                    agotados = df[df[col] == 0]
                    print(f"INFO: Hay {len(agotados)} productos con stock en 0.")

    print("\n" + "="*60)

In [203]:
analizar_estadisticas_numericas(config_tablas)

ANÁLISIS ESTADÍSTICO Y DETECCIÓN DE ANOMALÍAS

Tablas: PEDIDOS
        total_bruto  descuento_pct    total_neto
count  1.200000e+03    1200.000000  1.200000e+03
mean   1.761831e+06       0.058808  1.657728e+06
std    1.000344e+06       0.092635  9.551599e+05
min    2.750140e+04       0.000000  2.352719e+04
25%    9.011664e+05       0.000000  8.269738e+05
50%    1.784357e+06       0.000000  1.648489e+06
75%    2.602896e+06       0.110000  2.413205e+06
max    3.495290e+06       0.300000  3.490626e+06

Tablas: DETALLE_PEDIDOS
          cantidad  precio_unitario  descuento_pct      subtotal
count  4177.000000     4.177000e+03    4177.000000  4.177000e+03
mean      3.001197     6.831043e+05       0.029722  2.013879e+06
std       1.435807     4.888842e+05       0.055741  1.883665e+06
min       1.000000     2.606248e+04       0.000000  2.137123e+04
25%       2.000000     2.482562e+05       0.000000  5.490516e+05
50%       3.000000     6.235828e+05       0.000000  1.417991e+06
75%       4.0000

# Esta funcion se encarga de validar las tablas en sus fechas.

jemplo:
* Fecha_pedio es mayor a Fecha_entrega para la tabla de Pedidos
* Donde algunas de las fechas de algunas de las tablas supere la fecha actual

In [204]:
def validar_logica_temporal_total(config):
    print("VERIFICACIÓN DE INCONSISTENCIAS CRONOLÓGICAS")
    print("="*60)
    hay_errores_globales = False

    # 1. PEDIDOS: Entrega vs Pedido
    df_p = config['pedidos']['df']
    err_pedidos = df_p[pd.to_datetime(df_p['fecha_entrega']) < pd.to_datetime(df_p['fecha_pedido'])]
    if not err_pedidos.empty:
        print(f"TABLA PEDIDOS: {len(err_pedidos)} registros tienen fecha de entrega anterior a la de pedido.")
        hay_errores_globales = True

    # 2. CLIENTES: Registro vs Consentimiento (No puede haber consentimiento antes de existir)
    df_c = config['clientes']['df']
    err_clientes = df_c[pd.to_datetime(df_c['fecha_consentimiento']) < pd.to_datetime(df_c['fecha_registro'])]
    if not err_clientes.empty:
        print(f"TABLA CLIENTES: {len(err_clientes)} registros con consentimiento previo al registro.")
        hay_errores_globales = True

    # 3. FECHAS FUTURAS: Todas las tablas contra el tiempo real
    fecha_hoy = pd.Timestamp.now()
    for nombre, info in config.items():
        df = info['df']
        cols_fecha = [c for c in df.columns if any(p in c.lower() for p in ['fecha', 'timestamp'])]

        for col in cols_fecha:
            futuras = df[pd.to_datetime(df[col], errors='coerce') > fecha_hoy]
            if not futuras.empty:
                print(f"TABLA {nombre.upper()}: {len(futuras)} registros en columna '{col}' superan la fecha de hoy.")
                hay_errores_globales = True

    if not hay_errores_globales:
        print("Todas las fechas son lógicas y consistentes en todos los archivos.")

    print("="*60)


In [205]:
validar_logica_temporal_total(config_tablas)

VERIFICACIÓN DE INCONSISTENCIAS CRONOLÓGICAS
Todas las fechas son lógicas y consistentes en todos los archivos.


# Funcion para la validacion financiera para las tablas Pedidos, detalles pedidos y Productos

* Pedidos: verifica que el monto que el cliente terminó pagando (total_neto) sea exactamente el resultado de aplicar el descuento al valor inicial (total_bruto)
* Detalle_pedido: Verifica que el subtotal registrado coincida con el cálculo de la cantidad por el precio_unitario, aplicando el descuento_pct individual del producto
* Productos: Verifica que el costo no sea mayor o igual que el precio de venta, debido a que se necesita que el producto genere ganancias

In [206]:
def validar_coherencia_financiera_por_tabla(config):
    print("VALIDACIÓN DE LÓGICA FINANCIERA (INDIVIDUAL POR TABLA)")
    print("="*60)

    tol = 0.01  # Tolerancia para evitar falsos positivos por redondeo
    hay_errores = False

    for nombre_tabla, info in config.items():
        df = info['df']

        # --- CASO 1: TABLA PEDIDOS ---
        if nombre_tabla == 'pedidos':
            # Validar: Neto = Bruto * (1 - Descuento)
            neto_calculado = df['total_bruto'] * (1 - df['descuento_pct'])
            errores = df[(neto_calculado - df['total_neto']).abs() > tol]

            if not errores.empty:
                print(f"PEDIDOS: {len(errores)} registros con error en cálculo de Total Neto.")
                hay_errores = True
            else:
                print("PEDIDOS: Consistencia Bruto vs Neto verificada.")

        # --- CASO 2: TABLA DETALLE_PEDIDOS ---
        elif nombre_tabla == 'detalle_pedidos':
            # Validar: Subtotal = (Cantidad * Precio) * (1 - Descuento)
            sub_calculado = df['cantidad'] * df['precio_unitario'] * (1 - df['descuento_pct'])
            errores = df[(sub_calculado - df['subtotal']).abs() > tol]

            if not errores.empty:
                print(f"DETALLE_PEDIDOS: {len(errores)} líneas con subtotal incorrecto.")
                hay_errores = True
            else:
                print("DETALLE_PEDIDOS: Consistencia de subtotales por item verificada.")

        # --- CASO 3: TABLA PRODUCTOS ---
        elif nombre_tabla == 'productos':
            # Validar: Margen de ganancia lógico (Precio > Costo)
            errores = df[df['costo'] >= df['precio_venta']]

            if not errores.empty:
                print(f"PRODUCTOS: {len(errores)} items con costo mayor o igual al precio (Margen negativo).")
                hay_errores = True
            else:
                print("PRODUCTOS: Todos los productos tienen márgenes positivos.")

    if not hay_errores:
        print("\nCONCLUÍDO: Todos los archivos son financieramente coherentes en sus cálculos internos.")

    print("="*60)


In [207]:
validar_coherencia_financiera_por_tabla(config_tablas)

VALIDACIÓN DE LÓGICA FINANCIERA (INDIVIDUAL POR TABLA)
PEDIDOS: Consistencia Bruto vs Neto verificada.
DETALLE_PEDIDOS: Consistencia de subtotales por item verificada.
PRODUCTOS: Todos los productos tienen márgenes positivos.

CONCLUÍDO: Todos los archivos son financieramente coherentes en sus cálculos internos.


# Función que identifica y categoriza las columnas que contienen información sensible. Se divide en dos niveles de protección

In [208]:
def perfilar_datos_sensibles(config):
    print("PERFILAMIENTO Y CLASIFICACIÓN DE SEGURIDAD")
    print("="*60)

    # 1. PII (Personal Identifiable Information): Requieren CENSURA/MÁSCARA
    # Son datos que identifican directamente al individuo.
    terminos_pii = ['telefono', 'direccion', 'documento', 'apellido']

    # 2. SENSIBLES: Requieren HASH (SHA-256)
    # Datos que necesitamos para identificar registros únicos o comunicación, pero anonimizados.
    terminos_hash = ['nombre', 'email', 'proveedor', 'correo']

    # Excepciones (Catálogo público)
    excepciones = ['producto', 'categoria', 'sku', 'almacen']

    for nombre_tabla, info in config.items():
        df = info['df']
        cols = df.columns

        # Identificar para Censura (PII)
        a_censurar = [c for c in cols if any(t in c.lower() for t in terminos_pii)
                      and not any(e in c.lower() for e in excepciones)]

        # Identificar para Hash (Sensibles)
        a_hashear = [c for c in cols if any(t in c.lower() for t in terminos_hash)
                     and not any(e in c.lower() for e in excepciones)
                     and c not in a_censurar]

        if a_censurar or a_hashear:
            print(f"TABLA: {nombre_tabla.upper()}")
            if a_censurar:
                print(f"CRÍTICO (Masking): {a_censurar}")
            if a_hashear:
                print(f"SENSIBLE (Hash SHA-256): {a_hashear}")
            print("-" * 30)

    print("\nESTRATEGIA SILVER:")
    print("* Censura: Se reemplazarán por 'REDACTED' o máscaras (ej. *****).")
    print("* Hash: Se aplicará SHA-256 para mantener trazabilidad sin exponer el dato.")
    print("="*60)

In [209]:
perfilar_datos_sensibles(config_tablas)

PERFILAMIENTO Y CLASIFICACIÓN DE SEGURIDAD
TABLA: CLIENTES
CRÍTICO (Masking): ['apellido', 'telefono']
SENSIBLE (Hash SHA-256): ['nombre', 'email']
------------------------------
TABLA: PRODUCTOS
SENSIBLE (Hash SHA-256): ['proveedor_id', 'nombre_proveedor']
------------------------------

ESTRATEGIA SILVER:
* Censura: Se reemplazarán por 'REDACTED' o máscaras (ej. *****).
* Hash: Se aplicará SHA-256 para mantener trazabilidad sin exponer el dato.


## Conclusiones del Diagnóstico Inicial
Después de pasar todos los datasets por las pruebas de validación, ya tengo el mapa completo de la salud de nuestra información. Estos son los puntos clave para tener en cuenta:

1. Calidad e Integridad (Lo que se encontro)
* El tema de los Duplicados: Detecté 21 registros repetidos en la tabla de pedidos. Es un detalle técnico que voy a corregir de entrada en la siguiente fase para que los reportes de ventas sean exactos.

* Relaciones entre Tablas: La buena noticia es que la integridad es total. No hay datos "sueltos"; todos los pedidos y eventos están correctamente amarrados a sus clientes y productos.

* Espacios en Blanco: Tenemos algunos nulos en correos (4%) y teléfonos (8%). Nada del otro mundo, pero los voy a estandarizar para que la base de datos se vea uniforme.

2. Lógica y Finanzas (¿Todo cuadra?)
* Cuentas Claras: Validé los cálculos de cada tabla por separado y la matemática es consistente. El neto de los pedidos y los subtotales de los productos coinciden perfectamente con lo esperado.

* Cero Errores de Tiempo: Revisé las fechas y no hay inconsistencias. Todo sigue una línea de tiempo lógica: nadie recibió un pedido antes de comprarlo.

3. Seguridad y Privacidad (Gobernanza)
* Protección de Datos: Ya tengo clasificadas las columnas que no pueden quedar expuestas. Datos como apellidos y teléfonos se van a enmascarar por privacidad.

* Uso de Hash: Para nombres y correos, aplicaré un cifrado (Hash SHA-256). Así protegemos la identidad de las personas pero mantenemos la trazabilidad para los análisis de negocio.

### ¿Cuál es el siguiente paso?
Con este diagnóstico listo, lo que sigue es aplicar estas correcciones en la Capa Silver. Básicamente: limpiar duplicados, normalizar los campos vacíos y asegurar la privacidad de los datos sensibles